# CoALA Architecture -- Cognitive Architecture for Language Agents
## Customer Support Agent with Pinecone Memory + LangChain Tools

---

### What you will learn

- The CoALA control loop: Observe -> Parse -> Retrieve -> Reason -> Decide -> Act -> Learn
- **Semantic memory** -- facts stored and retrieved from Pinecone by vector similarity
- **Episodic memory** -- past conversation episodes stored in Pinecone and recalled when relevant
- **Procedural memory** -- skills stored as callable `@tool` functions, invoked directly when a matching intent is found
- How the **LangChain AgentExecutor** serves as Working Memory -- it receives all retrieved contexts and reasons over them
- How the **Learning phase** writes new knowledge back to all three memory stores after each interaction

## 1. CoALA Control Flow

```
USER MESSAGE
      |
      v
  [PARSE]             -- extract intent, order_id, sentiment
      |
      v
  [PARALLEL RETRIEVAL]
      |-----> Semantic Memory (Pinecone)   -- relevant facts
      |-----> Episodic Memory (Pinecone)   -- similar past episodes
      |-----> Procedural Memory (dict)     -- callable skill if intent known
      |
      v
  [WORKING MEMORY]    -- LangChain AgentExecutor reasons over all three contexts
      |
      v
  [DECISION PROCEDURE]
      |--> Procedural skill found?  YES -> call skill tool directly (fast path)
      |                             NO  -> agent reasons + calls tools (slow path)
      |
      v
  [ACTION / RESPONSE]
      |
      v
  [PARALLEL LEARNING]
      |-----> Semantic: store new facts to Pinecone
      |-----> Episodic: store episode summary to Pinecone
      |-----> Procedural: register tool if resolution was successful
```

## 2. Memory Store Design

| Store | Backend | What is stored | Retrieval method |
|---|---|---|---|
| Semantic | Pinecone `coala_semantic` namespace | General facts, policies | Vector similarity search |
| Episodic | Pinecone `coala_episodic` namespace | Past resolved conversations | Vector similarity search |
| Procedural | Python dict | Callable `@tool` functions (one per intent) | Exact intent key lookup |

In [ ]:
import os
import re
import json
import time
import pandas as pd
from typing import Optional
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

load_dotenv()

# LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# Embedding model (384-dim) -- same as memory_rag.ipynb
embedder = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# Order data
orders_df = pd.read_csv("order.csv")
print(f"Loaded {len(orders_df)} orders")
print(orders_df.head(3))

In [ ]:
# Pinecone setup
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

INDEX_NAME = "coala-memory"
DIM = 384
SEMANTIC_NS = "coala_semantic"
EPISODIC_NS = "coala_episodic"

existing = [d["name"] for d in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # wait for index to be ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

index = pc.Index(INDEX_NAME)
print(f"Index '{INDEX_NAME}' ready. Stats: {index.describe_index_stats()}")

## 3. Memory Store Classes

### Semantic Memory -- Pinecone `coala_semantic` namespace
Stores general facts (policies, domain knowledge). Retrieved by vector similarity to the current message.

In [ ]:
class SemanticMemory:
    """Stores general facts in Pinecone. Retrieved by semantic similarity."""

    def __init__(self, index, embedder, namespace=SEMANTIC_NS):
        self.index = index
        self.embedder = embedder
        self.namespace = namespace

    def learn(self, fact: str):
        vec = self.embedder.encode([fact], normalize_embeddings=True)[0].tolist()
        vid = f"fact_{int(time.time())}_{abs(hash(fact)) % 100000}"
        self.index.upsert(
            vectors=[(vid, vec, {"text": fact, "ts": int(time.time())})],
            namespace=self.namespace,
        )
        print(f"  [SemanticMemory.learn] stored: {fact[:60]}...")

    def retrieve(self, query: str, k: int = 3) -> list:
        qv = self.embedder.encode([query], normalize_embeddings=True)[0].tolist()
        res = self.index.query(
            vector=qv, top_k=k, include_metadata=True, namespace=self.namespace
        )
        hits = [m["metadata"]["text"] for m in res.get("matches", []) if m["score"] > 0.25]
        print(f"  [SemanticMemory.retrieve] {len(hits)} facts retrieved")
        return hits


class EpisodicMemory:
    """Stores past resolved episodes in Pinecone. Retrieved by semantic similarity to current situation."""

    def __init__(self, index, embedder, namespace=EPISODIC_NS):
        self.index = index
        self.embedder = embedder
        self.namespace = namespace

    def learn(self, episode: dict):
        summary = episode.get("summary", str(episode))
        vec = self.embedder.encode([summary], normalize_embeddings=True)[0].tolist()
        vid = f"ep_{int(time.time())}_{abs(hash(summary)) % 100000}"
        meta = {"summary": summary, "intent": episode.get("intent", ""),
                "outcome": episode.get("outcome", ""), "ts": int(time.time())}
        self.index.upsert(vectors=[(vid, vec, meta)], namespace=self.namespace)
        print(f"  [EpisodicMemory.learn] stored episode: {summary[:60]}...")

    def retrieve(self, query: str, k: int = 2) -> list:
        qv = self.embedder.encode([query], normalize_embeddings=True)[0].tolist()
        res = self.index.query(
            vector=qv, top_k=k, include_metadata=True, namespace=self.namespace
        )
        hits = [m["metadata"] for m in res.get("matches", []) if m["score"] > 0.25]
        print(f"  [EpisodicMemory.retrieve] {len(hits)} episodes retrieved")
        return hits


class ProceduralMemory:
    """Stores callable @tool procedures keyed by intent. Retrieved by exact intent match."""

    def __init__(self):
        self.skills = {}  # intent (str) -> LangChain tool

    def register(self, intent: str, tool_fn):
        self.skills[intent] = tool_fn
        print(f"  [ProceduralMemory.register] skill registered for intent: '{intent}'")

    def learn(self, intent: str, tool_fn):
        self.skills[intent] = tool_fn
        print(f"  [ProceduralMemory.learn] new skill learned for intent: '{intent}'")

    def retrieve(self, intent: str):
        skill = self.skills.get(intent)
        if skill:
            print(f"  [ProceduralMemory.retrieve] skill found for intent: '{intent}'")
        else:
            print(f"  [ProceduralMemory.retrieve] no skill for intent: '{intent}' -- will reason from scratch")
        return skill

## 4. Seed Memory Stores

Pre-load Semantic and Episodic memory with initial knowledge.
In production these would accumulate over time. Here we seed them once to give the agent a head start.

In [ ]:
semantic_mem = SemanticMemory(index, embedder)
episodic_mem = EpisodicMemory(index, embedder)
procedural_mem = ProceduralMemory()

# Seed Semantic Memory -- general facts and policies
print("Seeding Semantic Memory...")
semantic_facts = [
    "Orders delayed by more than 3 days qualify for a 10% compensation discount.",
    "Refunds are processed within 5-7 business days after the request is approved.",
    "Orders with status 'Processing' have not yet been dispatched to the courier.",
    "Orders with status 'Shipped' are in transit and cannot be cancelled.",
    "Cancelled orders can only be reversed within 24 hours of cancellation.",
    "Customers with more than one delayed order in 30 days receive priority support.",
]
for fact in semantic_facts:
    semantic_mem.learn(fact)

# Seed Episodic Memory -- past resolved conversations
print("\nSeeding Episodic Memory...")
past_episodes = [
    {
        "summary": "Customer reported order delay. Agent fetched order, confirmed Processing status, offered 10% discount. Customer satisfied.",
        "intent": "order_delay",
        "outcome": "10% compensation applied, customer satisfied",
    },
    {
        "summary": "Customer asked for order status. Agent retrieved order details and shipping status. Customer acknowledged.",
        "intent": "order_status",
        "outcome": "Shipping status provided, no further action needed",
    },
    {
        "summary": "Customer complained about cancellation. Agent escalated to human team. Representative resolved within 24 hours.",
        "intent": "order_cancellation",
        "outcome": "Escalated to human, resolved in 24 hours",
    },
]
for ep in past_episodes:
    episodic_mem.learn(ep)

print("\nMemory seeding complete.")

## 5. LangChain Tools -- Atomic Actions

These are the low-level tools available to both the Working Memory agent and the Procedural skills.

In [ ]:
@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)


@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    messages = {
        "Delivered":  "Your order has been delivered.",
        "Shipped":    "Your order is currently on the way.",
        "Processing": "Your order is still being prepared and has not shipped yet.",
        "Cancelled":  "Your order has been cancelled.",
    }
    return messages.get(status, "Order status unknown.")


@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."


@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."


@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return (
        f"The issue{suffix} has been escalated to the human support team. "
        "A representative will contact the customer within 24 hours."
    )


action_tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]

## 6. Procedural Memory -- Skills as Tools

Each procedural skill is a `@tool` function that executes a **pre-compiled sequence of atomic tools**.

When the agent encounters a familiar intent, it calls the skill directly -- no LLM reasoning needed.
This is the procedural memory equivalent of muscle memory: fast, reliable, no deliberation.

Skills are registered at startup and can also be **learned from successful interactions** (see Learning Phase).

In [ ]:
@tool
def skill_handle_order_delay(order_id: int) -> str:
    """
    Procedural skill: handle an order delay complaint.
    Steps: fetch order -> check shipping status -> offer 10% compensation.
    """
    print("  [PROCEDURAL SKILL] handle_order_delay -- running pre-compiled steps")
    order_raw  = fetch_order.invoke({"order_id": order_id})
    status_msg = check_shipping_status.invoke({"order_id": order_id})
    comp_msg   = offer_compensation.invoke({"order_id": order_id})
    try:
        record = json.loads(order_raw)
        name = record.get("user_name", "there")
        product = record.get("product_name", "your item")
    except Exception:
        name, product = "there", "your item"
    return (
        f"Hi {name}, regarding your {product} (Order #{order_id}): "
        f"{status_msg} {comp_msg}"
    )


@tool
def skill_handle_order_status(order_id: int) -> str:
    """
    Procedural skill: answer an order status enquiry.
    Steps: fetch order -> check shipping status -> provide info.
    """
    print("  [PROCEDURAL SKILL] handle_order_status -- running pre-compiled steps")
    order_raw  = fetch_order.invoke({"order_id": order_id})
    status_msg = check_shipping_status.invoke({"order_id": order_id})
    info_msg   = provide_order_info.invoke({"order_id": order_id})
    try:
        name = json.loads(order_raw).get("user_name", "there")
    except Exception:
        name = "there"
    return f"Hi {name}! {info_msg} {status_msg}"


@tool
def skill_handle_cancellation(order_id: Optional[int] = None) -> str:
    """
    Procedural skill: handle a cancellation complaint.
    Steps: fetch order (if ID available) -> escalate to human.
    """
    print("  [PROCEDURAL SKILL] handle_cancellation -- running pre-compiled steps")
    if order_id:
        fetch_order.invoke({"order_id": order_id})
    return escalate_to_human.invoke({"order_id": order_id})


# Register pre-compiled skills in Procedural Memory
procedural_mem.register("order_delay",        skill_handle_order_delay)
procedural_mem.register("order_status",       skill_handle_order_status)
procedural_mem.register("order_cancellation", skill_handle_cancellation)

## 7. PARSE Phase

All observations are converted to a structured internal representation before any memory retrieval or reasoning.
This is the **gateway** -- no raw text ever enters the memory or reasoning systems.

In [ ]:
def parse_observation(message: str) -> dict:
    """Convert raw user message into a structured internal representation."""
    msg = message.lower()

    # Extract order ID
    match = re.search(r'\b(50\d{2})\b', msg)
    order_id = int(match.group(1)) if match else None

    # Classify intent
    if any(w in msg for w in ["delay", "late", "not arrived", "not received", "not shipped"]):
        intent = "order_delay"
    elif any(w in msg for w in ["cancel", "cancellation", "cancelled"]):
        intent = "order_cancellation"
    elif any(w in msg for w in ["where", "status", "track", "when", "update", "details"]):
        intent = "order_status"
    else:
        intent = "general_enquiry"

    # Classify sentiment
    negative_words = ["unhappy", "angry", "frustrated", "unacceptable", "furious", "worst", "terrible"]
    sentiment = "negative" if any(w in msg for w in negative_words) else "neutral"

    parsed = {"raw": message, "intent": intent, "order_id": order_id, "sentiment": sentiment}
    print(f"  [PARSE] intent={intent}, order_id={order_id}, sentiment={sentiment}")
    return parsed

## 8. Working Memory -- LangChain Agent with Memory Context

Working Memory is the **only place where the LLM reasons**.
It receives all three retrieved memory contexts and uses them to decide which tools to call.
The system prompt is built dynamically from retrieved memory -- so each interaction is informed by past experience.

In [ ]:
def build_working_memory_agent(semantic_ctx: list, episodic_ctx: list) -> AgentExecutor:
    """Build a LangChain AgentExecutor whose system prompt is enriched with retrieved memory."""

    semantic_block = "\n".join(f"- {f}" for f in semantic_ctx) if semantic_ctx else "None available."
    episodic_block  = (
        "\n".join(
            f"- {e.get('summary','')} (outcome: {e.get('outcome','')})"
            for e in episodic_ctx
        )
        if episodic_ctx else "No similar past episodes."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""
You are a customer support agent with access to long-term memory.

--- SEMANTIC MEMORY (relevant facts and policies) ---
{semantic_block}

--- EPISODIC MEMORY (similar past resolutions) ---
{episodic_block}

Use the above memory to inform your reasoning. If a past episode suggests a successful resolution pattern, follow it.

To resolve the customer's issue:
1. Call fetch_order to get real order data
2. Call check_shipping_status to understand delivery status
3. Based on the customer's need, call one or more of: provide_order_info, offer_compensation, escalate_to_human

Compose an empathetic, personalised final response grounded in tool results and memory.
"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agent = create_tool_calling_agent(llm=llm, tools=action_tools, prompt=prompt)
    return AgentExecutor(agent=agent, tools=action_tools, verbose=True)

## 9. Learning Phase

After every interaction, the agent writes back to all three memory stores in parallel.
This is how CoALA agents improve over time -- each resolved interaction makes the next one cheaper and faster.

In [ ]:
def learning_phase(parsed: dict, response: str, success: bool,
                   sem_mem: SemanticMemory, ep_mem: EpisodicMemory, proc_mem: ProceduralMemory):
    """Update all three memory stores after a resolved interaction."""
    print("\n[LEARNING PHASE]")

    # Semantic: store the resolution as a new fact
    fact = f"For intent '{parsed['intent']}': successful resolution -- {response[:80]}"
    sem_mem.learn(fact)

    # Episodic: store the full episode
    ep_mem.learn({
        "summary": f"{parsed['raw'][:80]} -> {response[:80]}",
        "intent": parsed["intent"],
        "outcome": "resolved" if success else "unresolved",
        "order_id": str(parsed.get("order_id", "")),
    })

    # Procedural: if successful and a skill already exists, confirm it; otherwise note the gap
    if success and parsed["intent"] in proc_mem.skills:
        print(f"  [ProceduralMemory] skill for '{parsed['intent']}' confirmed -- no update needed")
    elif success:
        print(f"  [ProceduralMemory] no skill for '{parsed['intent']}' -- a new skill could be authored and registered here")

## 10. CoALA Agent Loop

In [ ]:
class CoALAAgent:

    def __init__(self, semantic: SemanticMemory, episodic: EpisodicMemory, procedural: ProceduralMemory):
        self.semantic   = semantic
        self.episodic   = episodic
        self.procedural = procedural

    def handle(self, message: str) -> str:
        print(f"\nUSER: {message}")
        print("="*60)

        # 1. PARSE
        print("\n[PARSE]")
        parsed = parse_observation(message)

        # 2. PARALLEL RETRIEVAL
        print("\n[PARALLEL RETRIEVAL]")
        semantic_ctx  = self.semantic.retrieve(message)
        episodic_ctx  = self.episodic.retrieve(message)
        procedural_fn = self.procedural.retrieve(parsed["intent"])

        # 3. DECISION PROCEDURE
        if procedural_fn and parsed.get("order_id"):
            # Fast path: known intent + order ID -> run procedural skill directly
            print("\n[DECISION] Procedural skill available -- skipping LLM reasoning")
            response = procedural_fn.invoke({"order_id": parsed["order_id"]})
        else:
            # Slow path: build Working Memory agent with retrieved context and reason
            print("\n[DECISION] No matching skill -- engaging Working Memory (LangChain agent)")
            agent_executor = build_working_memory_agent(semantic_ctx, episodic_ctx)
            result = agent_executor.invoke({"input": message, "chat_history": []})
            response = result["output"]

        # 4. LEARNING
        learning_phase(parsed, response, success=True,
                       sem_mem=self.semantic, ep_mem=self.episodic, proc_mem=self.procedural)

        print(f"\nFINAL RESPONSE:\n{response}")
        return response

## 11. Interactions

### Interaction 1 -- Known intent, procedural skill available (fast path)
The intent is `order_delay`. A skill is registered. No LLM call needed.

In [ ]:
agent = CoALAAgent(semantic_mem, episodic_mem, procedural_mem)

agent.handle("My order 5003 has been delayed and I am very unhappy. I want compensation.")

### Interaction 2 -- Unknown intent, Working Memory (slow path)
The intent is `general_enquiry` -- no procedural skill registered.
The LangChain agent reasons using semantic and episodic context retrieved from Pinecone.

In [ ]:
agent.handle("I need help with order 5005. It was cancelled but I never requested that.")

### Interaction 3 -- Order status enquiry (procedural skill)
The intent is `order_status`. Skill fires directly.

In [ ]:
agent.handle("Can you give me a status update on order 5020?")

## 12. Inspect Memory Stores

Query Pinecone directly to see what was stored after the interactions above.

In [ ]:
print("=== SEMANTIC MEMORY (sample query: order delay compensation) ===")
for fact in semantic_mem.retrieve("order delay compensation", k=5):
    print(" -", fact)

print("\n=== EPISODIC MEMORY (sample query: delayed order unhappy customer) ===")
for ep in episodic_mem.retrieve("delayed order unhappy customer", k=5):
    print(" - Summary:", ep.get("summary", "")[:80])
    print("   Outcome:", ep.get("outcome", ""))

print("\n=== PROCEDURAL MEMORY (registered skills) ===")
for intent, skill in procedural_mem.skills.items():
    print(f" - intent='{intent}' -> tool='{skill.name}'")

## 13. Key Takeaways

| Concept | Implementation |
|---|---|
| Semantic Memory | Pinecone `coala_semantic` namespace, vector similarity retrieval |
| Episodic Memory | Pinecone `coala_episodic` namespace, vector similarity retrieval |
| Procedural Memory | Python dict of intent -> `@tool` callables, exact key lookup |
| Working Memory | LangChain `AgentExecutor` with dynamically-built system prompt from retrieved context |
| Fast path | Procedural skill found + order ID known -> LLM never called |
| Slow path | No skill or no order ID -> full LangChain agent reasons from scratch |
| Learning | All three stores updated after every interaction |

**CoALA vs. BDI (Notebook_3) vs. Layered (Notebook_4):**

| Feature | BDI | Layered | CoALA |
|---|---|---|---|
| Persistent memory across sessions | No | No | Yes (Pinecone) |
| Learns from past interactions | No | No | Yes (all 3 stores) |
| Fast-path without LLM | No | Yes (reactive layer) | Yes (procedural skill) |
| Context-aware reasoning | Partial | Partial | Yes (semantic + episodic in prompt) |